# Generate background eddy time series - take mean and standard deviation of sargasso sea water and slope water for comparison to eddy

(as they did in Silver et al., 2021; Scientific Reports)

Author: SEL

In [1]:
import math, pylab, csv
import xarray as xr
import numpy as np
from datetime import datetime
from datetime import date
from itertools import groupby
from collections import Counter
from matplotlib.path import Path
import matplotlib.pyplot as plt
import earthaccess
import xarray as xr
from xarray.backends.api import open_datatree
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
#%matplotlib widget
from scipy.ndimage import generic_filter
from scipy.ndimage import gaussian_filter
from scipy.interpolate import griddata
import seaborn as sns 
import pandas
from xarray.backends.api import open_datatree
import numpy as np
import os

In [2]:
# %matplotlib widget

In [3]:
fontsize = 20

plt.rc('font', size=fontsize)          # controls default text sizes
plt.rc('axes', titlesize=fontsize)     # fontsize of the axes title
plt.rc('axes', labelsize=fontsize)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=fontsize)    # fontsize of the tick labels
plt.rc('ytick', labelsize=fontsize)    # fontsize of the tick labels
plt.rc('legend', fontsize=fontsize)    # legend fontsize
plt.rc('figure', titlesize=fontsize)  # fontsize of the figure title

In [17]:
moana_location='/home/jovyan/go-swace/data/moana/moana164996' #here, add your directory where all MOANA netCDF images are stored. Must precrop before running SeaDAS on Level-1 

In [5]:
# ds = xr.open_dataset('Edward_Eddy_trajectory_nrt_3.2exp_cyclonic_20180101_20240723.nc')
# ds

In [12]:
def crop_dataset_by_lat_lon(data,latmin,latmax,lonmin,lonmax):
    mask = (data.latitude >= latmin) & (data.latitude <= latmax) & (data.longitude >= lonmin) & (data.longitude <= lonmax) #Remove the 360
    mask=mask.assign_coords(longitude=coord_data['longitude'],latitude=coord_data['latitude'])
    ds_masked = data.where(mask, drop=True)
    return ds_masked

In [46]:
#define polygon
lonmin=290-360
lonmax=300-360
latmax=35
latmin=32.5

bbox_sargasso = (lonmin, latmin, lonmax, latmax)

In [47]:
#define polygon slope
lonmin=290-360
lonmax=298-360
latmax=42
latmin=38

bbox_slope = (lonmin, latmin, lonmax, latmax)

In [35]:
files=[]
for filename in os.listdir(moana_location):
    if filename.endswith('.nc'):
        files.append(filename)

In [19]:
files

['PACE_OCI.20240619T154945.L2_MOANA.V3.nc',
 'PACE_OCI.20240507T152746.L2_MOANA.V3.nc',
 'PACE_OCI.20240502T172807.L2_MOANA.V3.nc',
 'PACE_OCI.20240410T161150.L2_MOANA.V3.nc',
 'PACE_OCI.20240516T154654.L2_MOANA.V3.nc',
 'PACE_OCI.20240701T161253.L2_MOANA.V3.nc',
 'PACE_OCI.20240602T154850.L2_MOANA.V3.nc',
 'PACE_OCI.20240614T161245.L2_MOANA.V3.nc',
 'PACE_OCI.20240324T160553.L2_MOANA.V3.nc',
 'PACE_OCI.20240715T160621.L2_MOANA.V3.nc',
 'PACE_OCI.20240409T153648.L2_MOANA.V3.nc',
 'PACE_OCI.20240415T155017.L2_MOANA.V3.nc',
 'PACE_OCI.20240426T153655.L2_MOANA.V3.nc',
 'PACE_OCI.20240603T162339.L2_MOANA.V3.nc',
 'PACE_OCI.20240625T160122.L2_MOANA.V3.nc',
 'PACE_OCI.20240524T153042.L2_MOANA.V3.nc',
 'PACE_OCI.20240703T154355.L2_MOANA.V3.nc',
 'PACE_OCI.20240616T154355.L2_MOANA.V3.nc',
 'PACE_OCI.20240606T162940.L2_MOANA.V3.nc',
 'PACE_OCI.20240407T160505.L2_MOANA.V3.nc',
 'PACE_OCI.20240611T160650.L2_MOANA.V3.nc',
 'PACE_OCI.20240607T152607.L2_MOANA.V3.nc',
 'PACE_OCI.20240717T153717.L2_MO

In [55]:
picomean=[]
picostd=[]
promean=[]
prostd=[]
synmean=[]
synstd=[]
chlmean=[]
chlstd=[]
pocmean=[]
pocstd=[]
N=[]
satdate=[]

In [20]:
picomean1=[]
picostd1=[]
promean1=[]
prostd1=[]
synmean1=[]
synstd1=[]
chlmean1=[]
chlstd1=[]
pocmean1=[]
pocstd1=[]

In [21]:
picomean2=[]
picostd2=[]
promean2=[]
prostd2=[]
synmean2=[]
synstd2=[]
chlmean2=[]
chlstd2=[]
pocmean2=[]
pocstd2=[]

In [48]:
bbox=bbox_sargasso

In [49]:
bbox

(-70, 32.5, -60, 35)

In [60]:
for IDX in range(0,len(files)): #FOR EACH IMAGE. had to do in chunks 
    datatree = open_datatree(moana_location + '/' + files[IDX])
    dataset = xr.merge(datatree.to_dict().values())
    dataset = dataset.set_coords(("longitude", "latitude"))
    h=str(files[IDX])
    sat_date=h[9:17]
    mask = (dataset.longitude >= bbox[0]) & (dataset.longitude <= bbox[2]) & (dataset.latitude >= bbox[1]) & (dataset.latitude <= bbox[3])
    if mask.sum() == 0:
        mask = (dataset.longitude >= bbox[0]) & (dataset.longitude <= bbox[2])
        if mask.sum() == 0:
            print('Mask via longitude only')
            mask = (dataset.latitude >= bbox[1]) & (dataset.latitude <= bbox[3])
            print(mask.sum())
        else:
            print('Mask via latitude only')
            print(mask.sum())
    
    ds_masked = dataset.where(mask, drop=True)
    lat=ds_masked.latitude.values.flatten()
    lon=ds_masked.longitude.values.flatten()+360
    pico=ds_masked.picoeuk_moana.values.flatten()
    pro=ds_masked.prococcus_moana.values.flatten()
    syn=ds_masked.syncoccus_moana.values.flatten()
    chl=ds_masked.chlor_a.values.flatten()
    poc=ds_masked.poc.values.flatten()

    year=sat_date[0:4]
    month=sat_date[4:6]
    day=sat_date[6:8]

    time=year+'-'+month+'-'+day

    # for index in np.arange(0,len(lat)):
    #     val=in_eddy(ds,lat[index],lon[index],time)
    #     if val:
    #         pico_inside.append(pico[index]) 
    #         pro_inside.append(pro[index])
    #         syn_inside.append(syn[index])
    #         chl_inside.append(chl[index])
    #         poc_inside.append(poc[index])
    
    N1=np.size(np.where(np.isnan(pico)==False))

    if N1>5000:
        pico_mean=np.nanmean(pico);
        pico_std=np.nanstd(pico);
        pro_mean=np.nanmean(pro);
        pro_std=np.nanstd(pro);
        syn_mean=np.nanmean(syn);
        syn_std=np.nanstd(syn);
        chl_mean=np.nanmean(chl);
        chl_std=np.nanstd(chl);
        poc_mean=np.nanmean(poc);
        poc_std=np.nanstd(poc);

        chlmean.append(chl_mean);
        pocmean.append(poc_mean);
        picomean.append(pico_mean);
        promean.append(pro_mean);
        synmean.append(syn_mean);
        
        chlstd.append(chl_std);
        pocstd.append(poc_std);
        picostd.append(pico_std);
        prostd.append(pro_std);
        synstd.append(syn_std);
        N.append(N1);
        satdate.append(sat_date);

In [57]:
N1

36350

In [58]:
d = {'satdate': satdate, 'N':N, 'chlmean': chlmean, 'chlstd': chlstd, 'pocmean': pocmean, 'pocstd': pocstd,
    'picomean':picomean, 'picostd':picostd, 'promean':promean, 'prostd':prostd, 'synmean':synmean, 'synstd':synstd}
data = pandas.DataFrame(data=d)
data

,satdate,N,chlmean,chlstd,pocmean,pocstd,picomean,picostd,promean,prostd,synmean,synstd
0,20240619,21275,0.084657,0.031998,33.532948,7.641776,2153.868652,800.079224,260091.500000,76175.429688,21666.738281,10665.829102
1,20240507,11419,0.168231,0.079779,51.805187,16.168560,5546.152832,3789.876221,358416.187500,86534.164062,58383.289062,46765.957031
2,20240502,9971,0.122689,0.020337,46.774326,4.664521,2182.941895,658.961853,344383.468750,50366.617188,21583.021484,8739.415039
3,20240410,8506,0.217590,0.046724,64.821411,8.377510,8965.480469,2881.449707,456437.437500,54985.691406,111541.140625,43925.199219
4,20240516,8672,0.160425,0.061422,52.596470,12.163932,2913.140137,1278.607056,311808.000000,69253.023438,27170.033203,12585.424805
5,20240701,34724,0.066737,0.015496,34.851906,3.835403,1495.903076,348.161835,264745.843750,44838.914062,15369.333008,5194.270508
6,20240602,35056,0.116952,0.041235,39.888329,9.082416,2411.271729,825.985352,255465.734375,63883.902344,25823.806641,10861.696289
7,20240614,34737,0.065436,0.026423,33.579346,7.550807,1359.181152,429.708466,256235.468750,75318.664062,12542.134766,6678.483398
8,20240324,53237,0.238484,0.072375,68.099564,14.476791,6485.871582,3065.337402,464265.562500,66567.828125,112774.992188,76972.203125
9,20240715,48279,0.055806,0.009799,31.224953,4.213984,1328.120361,281.367767,247881.406250,51348.585938,12637.005859,4176.791016


In [ ]:
IDX

In [61]:
data_sorted = data.sort_values(by='satdate')

In [62]:
data_sorted

,satdate,N,chlmean,chlstd,pocmean,pocstd,picomean,picostd,promean,prostd,synmean,synstd
8,20240324,53237,0.238484,0.072375,68.099564,14.476791,6485.871582,3065.337402,464265.562500,66567.828125,112774.992188,76972.203125
18,20240407,19626,0.226433,0.075099,64.892914,16.056871,7803.083984,4041.361328,399899.375000,83131.828125,100832.156250,72866.429688
10,20240409,11754,0.190215,0.061479,55.221870,11.153825,9061.161133,4170.238281,402013.562500,75085.578125,110687.109375,85599.414062
3,20240410,8506,0.217590,0.046724,64.821411,8.377510,8965.480469,2881.449707,456437.437500,54985.691406,111541.140625,43925.199219
11,20240415,23125,0.219883,0.046046,61.703354,10.128841,5178.153320,2048.726807,407712.062500,48008.097656,62052.027344,39384.148438
22,20240418,6685,0.220680,0.042743,57.664375,9.535287,5202.455078,1526.022095,373183.968750,53077.695312,42129.085938,21617.884766
24,20240420,5037,0.136122,0.044907,44.525562,6.183071,6271.546387,3666.571533,403156.281250,61180.531250,69570.976562,44293.593750
12,20240426,15738,0.122670,0.048285,44.043591,8.804180,2853.129150,1526.287231,291852.062500,65436.117188,30975.357422,23147.179688
2,20240502,9971,0.122689,0.020337,46.774326,4.664521,2182.941895,658.961853,344383.468750,50366.617188,21583.021484,8739.415039
1,20240507,11419,0.168231,0.079779,51.805187,16.168560,5546.152832,3789.876221,358416.187500,86534.164062,58383.289062,46765.957031


In [63]:
data_sorted.to_csv('PACE_TIMESERIES_BACKGROUND_SARGASSO.csv')